In [172]:
ANON_MODE = True

if ANON_MODE:
    COL = {
        "PUB": "pub_anon_id",
        "AUTH": "author_anon_id",
        "FIRST": None,
        "LAST": None,
        "COUNTRY": "aff_country",
        "CCODE": "aff_country_code",
        "GENDER": "gender",
        "SPEC": "specialization",
    }
else:
    COL = {
        "PUB": "pub_id",
        "AUTH": None,              # not used
        "FIRST": "first_name",
        "LAST": "last_name",
        "COUNTRY": "aff_country",
        "CCODE": "aff_country_code",
        "GENDER": "gender",
        "SPEC": "specialization",
    }

UNKNOWN_GENDER = {None, "", "unknown", "Unknown", "UNK"}

In [173]:
class Author(object):
    """
    Supports BOTH:
    - non-anon mode: first/last names
    - anon mode: author_id (stable anonymized identifier)
    """
    def __init__(self, paper=None, first=None, last=None, country=None, country_code=None,
                 aff_city=None, gender=None, specialization=None, author_id=None):
        self.author_id = author_id  # anon id like Axxxx (or researcher_id)
        self.firstName = first
        self.lastName = last
        self.country = country
        self.country_code = country_code
        self.city = aff_city
        self.gender = gender
        self.paperList = []
        self.specialization = specialization
        if paper:
            self.paperList.append(paper)

    def key(self):
        # Unique identity key: prefer anon id, else name fallback
        if self.author_id is not None and str(self.author_id).strip() != "":
            return str(self.author_id)
        return (str(self.firstName).strip() + " " + str(self.lastName).strip()).strip()

    def display_name(self):
        # For printing tables: show anon id in anon mode, else human-readable name
        if self.author_id is not None and str(self.author_id).strip() != "":
            return str(self.author_id)
        return (str(self.firstName).strip() + " " + str(self.lastName).strip()).strip()

    def __eq__(self, other):
        return isinstance(other, Author) and self.key() == other.key()

    def __hash__(self):
        return hash(self.key())

In [174]:
def author_node_key(a):
    return a.key()

In [175]:
def author_from_row(row):
    if COL["AUTH"] is not None:
        return Author(
            row[COL["PUB"]],
            author_id=row[COL["AUTH"]],
            country=row.get(COL["COUNTRY"], None),
            country_code=row.get(COL["CCODE"], None),
            gender=row.get(COL["GENDER"], None),
            specialization=row.get(COL["SPEC"], None),
        )
    else:
        return Author(
            row[COL["PUB"]],
            first=row[COL["FIRST"]],
            last=row[COL["LAST"]],
            country=row.get(COL["COUNTRY"], None),
            country_code=row.get(COL["CCODE"], None),
            gender=row.get(COL["GENDER"], None),
            specialization=row.get(COL["SPEC"], None),
        )

In [176]:
def collectAuthorsOfOnePaper(df, pub_id, **kwargs):
    refAuthor = kwargs.get('refAuthor', None)
    authorList = []
    paper_df = df[df[COL["PUB"]] == pub_id]

    for _, row in paper_df.iterrows():
        a = author_from_row(row)
        if refAuthor is None or author_node_key(a) != author_node_key(refAuthor):
            authorList.append(a)
    return authorList

In [177]:
def searchAuthorPapers(df, author):
    paperDict = {}

    if COL["AUTH"] is not None:
        author_rows = df[df[COL["AUTH"]] == author.author_id]
    else:
        author_rows = df[(df[COL["FIRST"]] == author.firstName) & (df[COL["LAST"]] == author.lastName)]

    for pub_id in author_rows[COL["PUB"]].unique():
        paperDict[str(pub_id)] = collectAuthorsOfOnePaper(df, pub_id, refAuthor=author)

    return paperDict

In [178]:
import networkx as nx
def create_graph(collabDict, refAuthor=None):
    # collabDict: {paper_id: [Author, Author, ...]}
    dict_for_graph = {}
    for paper_id, authors in collabDict.items():
        dict_for_graph[paper_id] = [author_node_key(a) for a in authors]

    if refAuthor is not None:
        for k in dict_for_graph:
            dict_for_graph[k].append(author_node_key(refAuthor))

    return nx.from_dict_of_lists(dict_for_graph)

In [179]:
import math

def getReciprocal(n, d):
    if n != 0:
        return d/n
    else:
        return 0


def binaryCalculation(refAuthorFeature, collabFeature, baseFactor):
    if refAuthorFeature == collabFeature and baseFactor != 0:
        return 1/baseFactor
    else:
        return 0

def processCategoricalCalculation(collabCategory, baseFactor, countDict):
    if collabCategory not in countDict.keys():
        countDict[collabCategory] = baseFactor
    else:
        countDict[collabCategory] += baseFactor

def isWeightedCalculation(authorCategory, categoricalWeights):
    if categoricalWeights is not None:
        weightedCategories = {j for i in categoricalWeights.values() for j in i}
        if authorCategory in list(weightedCategories):
            return True
    return False

def concave_k(k: int, mode: str = "log") -> float:
    """
    Diminishing returns for number of unique categories (k).
    Anchored so k=1 -> 1.0 (i.e., no penalty for having only one category).
    """
    k = int(k)
    if k <= 1:
        return 1.0

    mode = (mode or "log").lower()
    if mode == "linear":
        return float(k)
    if mode == "sqrt":
        return math.sqrt(k)              # sqrt(1)=1
    if mode == "log":
        return 1.0 + math.log(k)         # 1 + log(1)=1, diminishing growth
    if mode in ("none", "off", "identity"):
        return float(k)

    raise ValueError(f"Unknown category_mode: {mode}")

def normalize_team(x: float, team_size: int, mode: str = "log") -> float:
    """
    Diminishing returns for team size.
    """
    team_size = int(team_size)
    if team_size <= 0:
        return x

    mode = (mode or "log").lower()
    if mode == "linear":
        return x / team_size
    if mode == "sqrt":
        return x / math.sqrt(team_size)
    if mode == "log":
        return x / math.log1p(team_size)
    if mode in ("none", "off"):
        return x

    raise ValueError(f"Unknown team_norm_mode: {mode}")

def calculateCIndex(author, collabDict, collabGraph, **kwargs):
    """
    Calculates the 'Community Index' (C-Index) for a single author.

    This index is an aggregation of diversity scores across three dimensions:
    gender, nationality, and specialization. The score for each paper is calculated,
    and the final index is the average across all of an author's papers.

    The calculation can optionally include a bonus for collaborating with "new" authors,
    defined as authors who have only appeared on one paper in the reference author's network.

    Args:
        author (Author): The reference author for whom the index is calculated.
        collabDict (dict): The author's collaboration dictionary from searchAuthorPapers.
        collabGraph (nx.Graph): The author's collaboration network graph.
        **kwargs:
            crossPaper (bool): If True, apply a bonus for new collaborators.
            newBonus (float): The bonus multiplier for new collaborators.
            baseGenderFactor (float): Base weight for gender diversity calculation.
            baseNationalityBonus (float): Base weight for nationality diversity.
            baseSpecializationFactor (float): Base weight for specialization diversity.
            categoricalWeights (dict): Optional weights for specific countries or fields.
            normalization_mode (str): Select mode to normalize for increasing n authors. Default: "log" using ln(1+x). 

    Returns:
        tuple: (rounded final index, unrounded final index, detailed report list)
    """
    # --- Parameter Setup ---
    crossPaper = kwargs.get('crossPaper', False)
    newBonus = kwargs.get('newBonus', 0.8)
    # The 'isNew' parameter defines a "new" collaborator by their degree in the graph.
    # A degree of 1 means they only appear on one paper in this author's network.
    isNew = 1 

    # Normalization parameters
    category_mode = kwargs.get("category_mode", "log")          # diminishing returns for #categories
    team_norm_mode = kwargs.get("team_norm_mode", "log")        # diminishing returns for team size
    apply_team_norm = kwargs.get("apply_team_norm", True)       # toggle

    # Base factors for diversity calculations
    baseGenderFactor = kwargs.get('baseGenderFactor', 1)
    baseNationalityBonus = kwargs.get('baseNationalityBonus', 1)
    baseSpecializationFactor = kwargs.get('baseSpecializationFactor', 1)
    categoricalWeights = kwargs.get('categoricalWeights', None)

    paperFeatureIndices = []
    paper_details_for_reporting = [] 

    # --- Calculation Loop: Iterate through each paper the author has written ---
    for publication in collabDict.keys():
        # Initialize diversity scores for this specific paper
        # Gender: Starts with the author's own contribution
        genderFactor = 1 * baseGenderFactor
        # Nationality/Specialization: Track counts of each category
        nationalityCounts = {author.country_code : 1 * baseNationalityBonus}
        specializationCounts = {author.specialization : 1 * baseSpecializationFactor}

        # --- Loop through collaborators on this paper ---
        for collab in collabDict[publication]:
            bonus = 1
            collab_key = author_node_key(collab)
            # Apply a bonus if the collaborator is "new" to the network            
            if crossPaper and collabGraph.degree[collab_key] == isNew:
                bonus += newBonus
            
            # --- Update diversity scores based on this collaborator ---
            # Gender: Adds to the score if genders are different
            if collab.gender not in UNKNOWN_GENDER:
                genderFactor += binaryCalculation(author.gender, collab.gender, baseGenderFactor*bonus)
            # Nationality & Specialization: Tally the counts for each category
            processCategoricalCalculation(collab.country_code,
                                          baseNationalityBonus*bonus,
                                          nationalityCounts)
            processCategoricalCalculation(collab.specialization,
                                          baseSpecializationFactor*bonus,
                                          specializationCounts)

        # The logic combines two ideas:
        # 1. Variety: The number of unique categories (e.g., `len(set(nationalityCounts.keys()))`)
        # 2. Balance: A weight that rewards even distribution and penalizes self-concentration.

        # Gender Factor: A simple reciprocal of the sum.       
        known_gender_collabs = sum(
            1 for c in collabDict[publication]
            if c.gender not in UNKNOWN_GENDER
        )
        final_gender_factor = getReciprocal(genderFactor, max(known_gender_collabs, 1))

        # Nationality Factor        
        nationality_denominator = sum(nationalityCounts.values()) - baseNationalityBonus
        nationality_weight = getReciprocal(nationalityCounts.get(author.country_code, 0), nationality_denominator)
        k_nat = len(set(nationalityCounts.keys()))
        final_nationality_factor = concave_k(k_nat, mode=category_mode) * nationality_weight
        
        # Specialization Factor
        specialization_denominator = sum(specializationCounts.values()) - baseSpecializationFactor
        specialization_weight = getReciprocal(specializationCounts.get(author.specialization, 0), specialization_denominator)
        k_spec = len(set(specializationCounts.keys()))
        final_specialization_factor = concave_k(k_spec, mode=category_mode) * specialization_weight        
        
        # Apply any specific external weights (e.g., for underrepresented countries)
        if isWeightedCalculation(author.country_code, categoricalWeights):
            final_nationality_factor *= categoricalWeights["nationality"][author.country_code]
        if isWeightedCalculation(author.specialization, categoricalWeights):
            final_specialization_factor *= categoricalWeights["specialization"][author.specialization]
        
        # The index for this paper is the sum of the three diversity factors
        paper_index = final_gender_factor + final_nationality_factor + final_specialization_factor
    
        # optional team-size normalization
        team_size = len(collabDict[publication]) + 1
        if apply_team_norm:
            paper_index = normalize_team(paper_index, team_size, mode=team_norm_mode)

        paperFeatureIndices.append(paper_index)
        
        # Store detailed results for reporting
        paper_details_for_reporting.append({
            "pub_id": f"pub.{publication}",
            "Gender Factor": final_gender_factor,
            "Nationality Factor": final_nationality_factor,
            "Specialization Factor": final_specialization_factor,
            "Paper Index": paper_index
        })
    
    # --- Aggregate across all papers ---
    # The final C-Index is the average of the paper indices.        
    unrounded_index = sum(paperFeatureIndices) / len(paperFeatureIndices)
    final_index = round(unrounded_index)
    return final_index, unrounded_index, paper_details_for_reporting

In [180]:
import pandas as pd

def generate_table_for_author_cohort(
    author_cohort,
    df,
    *,
    category_mode="log",
    apply_team_norm=True,
    team_norm_mode="log",
):
    """
    Generates a single, author-centric table for a specific, predefined
    list of author objects, showing all papers for each of them.

    Produces two author-level indices:
      - C-index   (baseline, no new-author bonus)
      - C-index+  (with new-author bonus)

    The per-paper factors shown (Country/Gender/Field/Paper factor) come from the BONUS run
    """
    all_rows_data = []

    print(
        f"Building table for a cohort of {len(author_cohort)} authors "
        f"(category_mode={category_mode}, apply_team_norm={apply_team_norm}, team_norm_mode={team_norm_mode})..."
    )

    for author in author_cohort:
        papers_dict = searchAuthorPapers(df, author)
        if not papers_dict:
            continue

        graph = create_graph(papers_dict)

        # baseline (no bonus)
        _, c_index_baseline_exact, _ = calculateCIndex(
            author,
            papers_dict,
            graph,
            crossPaper=False,
            category_mode=category_mode,
            apply_team_norm=apply_team_norm,
            team_norm_mode=team_norm_mode,
        )

        # bonus (new-author bonus)
        _, c_index_bonus_exact, details_bonus = calculateCIndex(
            author,
            papers_dict,
            graph,
            crossPaper=True,
            category_mode=category_mode,
            apply_team_norm=apply_team_norm,
            team_norm_mode=team_norm_mode,
        )

        for paper_details in details_bonus:
            row_data = {
                "Pub id": paper_details["pub_id"].replace("pub.", ""),
                "Name": author.display_name(),
                "Country": author.country,
                "Gender": "M" if author.gender == "male" else "F",
                "Field": author.specialization.replace("Science", "Sci").replace("Healthcare", "Health"),
                "C-index+": c_index_bonus_exact,
                "C-index": c_index_baseline_exact,
                "Paper factor": paper_details["Paper Index"],
                "Country factor": paper_details["Nationality Factor"],
                "Gender factor": paper_details["Gender Factor"],
                "Field factor": paper_details["Specialization Factor"],
            }
            all_rows_data.append(row_data)

    if not all_rows_data:
        print("No data to generate.")
        return None

    final_df = pd.DataFrame(all_rows_data)

    # Sort + reorder columns (as you like)
    final_df.sort_values(by=["Name", "Pub id"], inplace=True)
    final_df = final_df[
        [
            "Pub id",
            "Name",
            "Country",
            "Gender",
            "Field",
            "C-index+",
            "C-index",
            "Paper factor",
            "Country factor",
            "Gender factor",
            "Field factor",
        ]
    ]

    styled_df = (
        final_df.style
        .format({
            "C-index+": "{:.2f}",
            "C-index": "{:.2f}",
            "Country factor": "{:.2f}",
            "Gender factor": "{:.2f}",
            "Field factor": "{:.2f}",
            "Paper factor": "{:.2f}",
        })
        .set_properties(**{"text-align": "left"})
        .set_table_styles([dict(selector="th", props=[("text-align", "left")])])
    )

    return styled_df

In [181]:
# -----------------------------
# 1) Load
# -----------------------------
def load_cohort(csv_path: str) -> pd.DataFrame:
    """Load a cohort CSV."""
    return pd.read_csv(csv_path)

# -----------------------------
# 2) Exploration
# -----------------------------
def summarize_by_publication(df: pd.DataFrame) -> pd.DataFrame:
    """Summary table by pub_anon_id."""
    return (
        df.groupby("pub_anon_id")
          .agg(
              authors=("author_anon_id", "nunique"),
              countries=("aff_country", "nunique"),
              genders=("gender", "nunique"),
              specializations=("specialization", "nunique")
          )
          .sort_values("authors", ascending=False)
    )

def pubs_per_author(df: pd.DataFrame) -> pd.Series:
    """How many unique pubs each author appears on."""
    return df.groupby("author_anon_id")["pub_anon_id"].nunique().sort_values(ascending=False)

# -----------------------------
# 3) Build ego-network cohort from a seed author
# -----------------------------
def cohort_from_seed_author(df: pd.DataFrame, seed_author_id: str) -> pd.DataFrame:
    """
    Ego network cohort:
    seed author -> all their papers -> all authors on those papers.
    """
    seed_pubs = df.loc[df["author_anon_id"] == seed_author_id, "pub_anon_id"].unique()
    return df[df["pub_anon_id"].isin(seed_pubs)].copy()

# -----------------------------
# 4) Build Author objects from cohort df
# -----------------------------
def build_author_objects_from_df(cohort_df: pd.DataFrame):
    """
    Build one Author object per unique author_anon_id.
    Assumes your Author constructor supports:
      Author(paper=None, author_id=..., country=..., country_code=..., aff_city=..., gender=..., specialization=...)
    """
    author_objs = []
    for author_id, rows in cohort_df.groupby("author_anon_id"):
        row = rows.iloc[0]
        author_objs.append(
            Author(
                paper=None,
                author_id=author_id,
                country=row.get("aff_country", None),
                country_code=row.get("aff_country_code", None),
                aff_city=row.get("aff_city", None),
                gender=row.get("gender", None),
                specialization=row.get("specialization", None),
            )
        )
    return author_objs

# -----------------------------
# 5) End-to-end runner (explore + pick seed + table)
# -----------------------------
def run_cohort_workflow(
    csv_path: str,
    seed_author_id: str,
    label: str = ""
):
    """
    Runs:
      - load
      - pub summary
      - pubs-per-author
      - ego-network from seed author
      - build Author objects
      - generate cohort table
    Returns (df, summary_pub, pubs_by_author, cohort_df, author_objs, final_table)
    """
    df = load_cohort(csv_path)

    summary_pub = summarize_by_publication(df)
    pubs_by_author = pubs_per_author(df)

    cohort_df = cohort_from_seed_author(df, seed_author_id)
    author_objs = build_author_objects_from_df(cohort_df)

    print(f"\n--- {label or csv_path} ---")
    print("Seed author:", seed_author_id)
    print("Ego-network pubs:", cohort_df["pub_anon_id"].nunique())
    print("Ego-network authors:", cohort_df["author_anon_id"].nunique())
    print("Author objects built:", len(author_objs))

    final_table = generate_table_for_author_cohort(author_objs, cohort_df)

    return df, summary_pub, pubs_by_author, cohort_df, author_objs, final_table

In [182]:
# Cohort A
df_A, summary_A, pubs_A, cohortA_df, authorA_objs, table_A = run_cohort_workflow(
    csv_path="cohortA_public_demo.csv",
    seed_author_id="A38327606686f",
    label="Cohort A"
)

display(summary_A)
display(pubs_A)
display(table_A)


--- Cohort A ---
Seed author: A38327606686f
Ego-network pubs: 4
Ego-network authors: 12
Author objects built: 12
Building table for a cohort of 12 authors (category_mode=log, apply_team_norm=True, team_norm_mode=log)...


,authors,countries,genders,specializations
pub_anon_id,,,,
P7c82ccbe194a,20,4,3,5
P17d0ee011aa1,12,3,3,6
Pc97bf25865b0,7,2,2,4
P8d6cc990644e,6,2,2,3
Pc31ee0fc09c6,6,2,2,4
Pf8c66ca7c94a,6,3,1,3
Paeb9d85274a2,4,2,1,3


author_anon_id
A38327606686f    4
Ab1e8b19bae0b    4
Afb5914645166    3
Af103ef0d80ce    3
A11460d5aa670    3
Aa6270b01971b    2
A60af1130add6    2
A6e2aff8d4f11    2
A27d96bd66b84    1
Ad2c64d944b4d    1
A91b68983525e    1
A97ac53e456d9    1
A116dd43f5546    1
A0d87c26033d1    1
Ab6556211059e    1
Ab7464adbc9d2    1
Abd4f9c40c87e    1
Ad5ecc52b7111    1
A31be42760a0b    1
Ade11f80cd814    1
Ae75c59644b48    1
A0a4539517830    1
Af55d88d046b1    1
Af8ecffc8330e    1
Af9b4a2748a79    1
Afafd8838d7fe    1
A915c95ef1178    1
A77afe8504df7    1
A75e56c11dcda    1
A7405c4069613    1
A17cffab5e939    1
A3b3c93f5c45e    1
A3b590d2596a1    1
A4028428139e7    1
A441f32c5146e    1
A4655057dee62    1
A495ed70d748b    1
A4a0998b8e22b    1
A4de916b6f0b8    1
A52334e383e85    1
A5d5fe6cddb0d    1
A5d615cf56904    1
A163297e46e52    1
A03752a964def    1
A72195b54cd22    1
A0122def5b1c2    1
Name: pub_anon_id, dtype: int64

,Pub id,Name,Country,Gender,Field,C-index+,C-index,Paper factor,Country factor,Gender factor,Field factor
0,Pc31ee0fc09c6,A11460d5aa670,China,M,Materials Sci,4.30,3.57,2.55,1.69,0.92,3.27
1,Pc97bf25865b0,A11460d5aa670,China,M,Materials Sci,4.30,3.57,6.06,1.69,0.88,10.02
2,Paeb9d85274a2,A38327606686f,Spain,M,Environmental Sci,15.19,11.69,7.53,5.08,0.75,6.30
4,Pc31ee0fc09c6,A38327606686f,Spain,M,Environmental Sci,15.19,11.69,18.82,17.61,0.92,24.82
5,Pc97bf25865b0,A38327606686f,Spain,M,Environmental Sci,15.19,11.69,15.29,12.87,0.80,18.14
3,Pf8c66ca7c94a,A38327606686f,Spain,M,Environmental Sci,15.19,11.69,19.12,19.31,1.15,19.31
6,Pc31ee0fc09c6,A3b3c93f5c45e,China,F,Materials Sci,5.68,5.48,5.68,1.81,7.00,3.67
7,Pf8c66ca7c94a,A495ed70d748b,China,M,Materials Sci,3.70,3.14,3.70,2.76,1.38,3.54
8,Pc97bf25865b0,A5d5fe6cddb0d,China,F,Chemistry,5.98,5.51,5.98,1.83,5.00,5.60
9,Pc31ee0fc09c6,A60af1130add6,China,M,Environmental Engineering,16.30,9.40,16.30,1.79,1.38,34.36


In [183]:
# Cohort B (pick one of your 2-paper authors)
df_B, summary_B, pubs_B, cohortB_df, authorB_objs, table_B = run_cohort_workflow(
    csv_path="cohortB_public_demo.csv",
    seed_author_id="A77c21dacbd16",   # or A1d51b8d97097
    label="Cohort B"
)

display(summary_B)
display(pubs_B)
display(table_B)


--- Cohort B ---
Seed author: A77c21dacbd16
Ego-network pubs: 2
Ego-network authors: 23
Author objects built: 23
Building table for a cohort of 23 authors (category_mode=log, apply_team_norm=True, team_norm_mode=log)...


,authors,countries,genders,specializations
pub_anon_id,,,,
P12c4680a5a2f,18,5,2,6
P8180aac5f045,15,7,2,8
P67694f6e606a,9,9,2,2


author_anon_id
A77c21dacbd16    2
A1d51b8d97097    2
A05e14d24db0c    1
Ac047bf941cb8    1
A8562ad162ed5    1
A98687b84a2af    1
Aab3d61cb0d48    1
Ab03da4a3d8c5    1
Ab33d651ea75d    1
Abeff547e0117    1
Ac348b40d377b    1
A7377c4c6ea73    1
Ad1f9700c44cd    1
Ad75c9bd62613    1
Ad83bc45d319d    1
Ae4b5a5839b85    1
Aecf877d9eda3    1
Af46fd65cdf0f    1
Af64b60959415    1
A74828677caf2    1
A7103d6c4eb0b    1
A07510fbcac2b    1
A23e36e9365a6    1
A0a982b89120d    1
A0bde5634ba80    1
A14415b8c5b51    1
A16adb6f52b24    1
A17303fc2a02f    1
A1b6c01391205    1
A1e620567be50    1
A302195fc7c86    1
A6fe5a7c4f575    1
A3bbd0f5b0c45    1
A4968dbdd6ce3    1
A50f2943dbf49    1
A52cb276acc8b    1
A54200dd73478    1
A597692134005    1
A65cd1d1f01bb    1
Af751bd9fe6fe    1
Name: pub_anon_id, dtype: int64

,Pub id,Name,Country,Gender,Field,C-index+,C-index,Paper factor,Country factor,Gender factor,Field factor
0,P8180aac5f045,A05e14d24db0c,Denmark,M,Internal Medicine,35.62,20.55,35.62,100.75,2.16,5.54
1,P8180aac5f045,A07510fbcac2b,China,F,Finance,72.18,40.28,72.18,106.05,6.21,110.86
2,P8180aac5f045,A0a982b89120d,United States,F,Epidemiology,40.10,23.34,40.10,6.89,6.21,110.86
3,P8180aac5f045,A0bde5634ba80,United States,M,Internal Medicine,4.65,4.18,4.65,6.89,2.14,5.33
4,P8180aac5f045,A14415b8c5b51,Germany,M,Business,41.92,25.09,41.92,16.57,2.14,110.86
5,P67694f6e606a,A16adb6f52b24,Uganda,M,Internal Medicine,26.70,15.23,26.70,63.31,3.41,1.76
6,P8180aac5f045,A17303fc2a02f,Japan,M,Internal Medicine,36.73,21.13,36.73,106.05,2.14,5.33
7,P8180aac5f045,A1b6c01391205,United States,F,Internal Medicine,5.96,5.07,5.96,6.89,6.21,5.33
8,P8180aac5f045,A1d51b8d97097,Germany,M,Internal Medicine,9.73,8.29,9.73,21.90,2.16,5.54
9,P8180aac5f045,A1e620567be50,United States,M,Internal Medicine,4.96,4.46,4.96,7.41,2.16,5.54
